In [1]:
import pandas as pd
from ingest import build_index, load_faq_data
from tqdm.auto import tqdm

In [2]:
df_ground_truth = pd.read_csv("data/ground-truth-data.csv")

In [3]:
ground_truth = df_ground_truth.to_dict("records")

In [4]:
documents_llm = load_faq_data()

In [5]:
index = build_index(documents_llm)

In [6]:
gt_ids = {q["document"] for q in ground_truth}
index_ids = {doc["id"] for doc in documents_llm}
print(f"missing: {len(gt_ids - index_ids)} / {len(gt_ids)}")

missing: 6 / 1244


In [7]:
index.search("What is the course about?")

[{'id': '9e508f2212',
  'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: When does the course start?',
  'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."},
 {'id': 'bfafa427b3',
  'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: What are the prerequisites for this course?',
  'answer': "To get the most out of this course, you should have:\n\n- Basic coding experience\n- Familiarity with SQL\n- Experience with Python (helpful 

In [8]:
def text_search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    
    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [9]:
q = ground_truth[1]

In [10]:
q

{'question': 'Where can I find the homework assignments and the submission form for this cohort?',
 'document': '0e38656cfb'}

In [11]:
results = text_search(q["question"])
results

[{'id': '53f15299b6',
  'course': 'llm-zoomcamp',
  'section': 'Module 1 Homework',
  'question': 'Where can I find the homework questions?',
  'answer': 'Homework links are available in the course GitHub repo and in the course management platform.\n\nFor the 2026 Module 1 homework, use:\n\n- [Module 1 cohort materials](https://github.com/DataTalksClub/llm-zoomcamp/tree/main/cohorts/2026/01-agentic-rag)\n- [Module 1 homework instructions](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/cohorts/2026/01-agentic-rag/homework.md)\n\nThe course platform is useful for submission and deadlines, but the GitHub homework instructions often contain important extra context.'},
 {'id': '0190b37350',
  'course': 'mlops-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Where do I find the updated homework playlist and cohort-specific info for the MLOps course?',
  'answer': "All cohort-specific information (homework links, video playlist, deadlines, Slack channel) liv

In [12]:
doc_id = "53f15299b6"

In [13]:
for d in results:
    print(f"{d["id"]} == {doc_id}: {d["id"] == doc_id}")

53f15299b6 == 53f15299b6: True
0190b37350 == 53f15299b6: False
a9353fadfe == 53f15299b6: False
90b9e103c7 == 53f15299b6: False
50ebc48eea == 53f15299b6: False


In [14]:
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query = q["question"])
    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))
    
    return relevance

In [15]:
compute_relevance_text(q)

[0, 0, 0, 0, 0]

In [16]:
q = ground_truth[15]
compute_relevance_text(q)

[0, 1, 0, 0, 0]

In [17]:
q = ground_truth[70]
compute_relevance_text(q)

[1, 0, 0, 0, 0]

In [18]:
def compute_relevance_total_text(ground_truth):
    relevance_total = []
    for q in tqdm(ground_truth):
        relevance_total.append(compute_relevance_text(q))
        
    return relevance_total

In [19]:
relevance = compute_relevance_total_text(ground_truth)

  0%|          | 0/6220 [00:00<?, ?it/s]

In [20]:
sample = relevance[:15]

In [21]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query = q["question"])
    relevance = []
    
    for d in results:
        relevance.append(int(d["id"] == doc_id))
    
    return relevance

In [22]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []
    for q in tqdm(ground_truth):
        relevance_total.append(compute_relevance(q, search_function))
        
    return relevance_total

In [23]:
def hit_rate(relevance):
    cnt = 0
    for line in relevance:
        if 1 in line:
            cnt += 1
            
    return cnt / len(relevance)

In [24]:
hit_rate(sample)

0.5333333333333333

In [25]:
def mean_reciprocal_rank(relevance: list[list[int]]) -> float:
    total_rr = 0.0
    for line in relevance:
        if 1 in line:
            total_rr += 1 / (line.index(1) + 1)
        # else: contributes 0, but still counts toward the denominator
    return total_rr / len(relevance)

In [26]:
mean_reciprocal_rank(sample)

0.38333333333333336

In [29]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mean_reciprocal_rank(relevance_total)
    }

In [27]:
def text_search_v2(query):
    boost_dict = {"question": 2.0, "section": 0.5}
    return index.search(
        query,
        num_results = 5,
        boost_dict=boost_dict
    )

In [30]:
evaluate(ground_truth, text_search_v2)

  0%|          | 0/6220 [00:00<?, ?it/s]

{'hit_rate': 0.754983922829582, 'mrr': 0.6172293676312982}

In [31]:
def search_boost(query, question_boost):
    boost_dict = {"question": question_boost, 
                  "section": 0.5}
    
    return index.search(
        query,
        num_results = 5,
        boost_dict=boost_dict
    )

In [32]:
for boost in [1.0, 3.0, 5.0, 10.0]:
    result = evaluate(
        ground_truth,
        lambda query, boost=boost: search_boost(query, boost)
    )
    print(f"boost = {boost}: {result}")

  0%|          | 0/6220 [00:00<?, ?it/s]

boost = 1.0: {'hit_rate': 0.7598070739549839, 'mrr': 0.6244989281886405}


  0%|          | 0/6220 [00:00<?, ?it/s]

boost = 3.0: {'hit_rate': 0.7308681672025723, 'mrr': 0.5932020364415875}


  0%|          | 0/6220 [00:00<?, ?it/s]

boost = 5.0: {'hit_rate': 0.6913183279742765, 'mrr': 0.5595900321543419}


  0%|          | 0/6220 [00:00<?, ?it/s]

boost = 10.0: {'hit_rate': 0.6553054662379422, 'mrr': 0.5289844587352636}


In [33]:
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "answer": answer_boost,
        "section": section_boost
    }
    return index.search(
        query,
        num_results = 5,
        boost_dict = boost_dict
    )

In [34]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(f"Evaluating question_boost={question_boost}, answer_boost={answer_boost}, section_boost={section_boost}...")
            result = evaluate(
                ground_truth,
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query,
                    question_boost,
                    answer_boost,
                    section_boost
                )
            )

            results.append({
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.1...


  0%|          | 0/6220 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.2...


  0%|          | 0/6220 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.5...


...

  0%|          | 0/6220 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.2...


  0%|          | 0/6220 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.5...


  0%|          | 0/6220 [00:00<?, ?it/s]

In [35]:
results

[{'question': 1.0,
  'answer': 1.0,
  'section': 0.1,
  'hit_rate': 0.8467845659163987,
  'mrr': 0.7051500535905678},
 {'question': 1.0,
  'answer': 1.0,
  'section': 0.2,
  'hit_rate': 0.840192926045016,
  'mrr': 0.6974303322615221},
 {'question': 1.0,
  'answer': 1.0,
  'section': 0.5,
  'hit_rate': 0.7598070739549839,
  'mrr': 0.6244989281886405},
 {'question': 1.0,
  'answer': 2.0,
  'section': 0.1,
  'hit_rate': 0.8765273311897106,
  'mrr': 0.7333172561629152},
 {'question': 1.0,
  'answer': 2.0,
  'section': 0.2,
  'hit_rate': 0.8752411575562701,
  'mrr': 0.7320632368703108},
 {'question': 1.0,
  'answer': 2.0,
  'section': 0.5,
  'hit_rate': 0.840032154340836,
  'mrr': 0.7017952840300108},
 {'question': 1.0,
  'answer': 4.0,
  'section': 0.1,
  'hit_rate': 0.8610932475884244,
  'mrr': 0.7201312968917473},
 {'question': 1.0,
  'answer': 4.0,
  'section': 0.2,
  'hit_rate': 0.8609324758842444,
  'mrr': 0.7209163987138266},
 {'question': 1.0,
  'answer': 4.0,
  'section': 0.5,
  'h

In [36]:
df_results = pd.DataFrame(results)

In [37]:
df_results.sort_values("mrr", ascending=False).head(10)

,question,answer,section,hit_rate,mrr
3,1.0,2.0,0.1,0.876527,0.733317
19,2.0,4.0,0.2,0.876527,0.733317
35,5.0,10.0,0.5,0.876527,0.733317
34,5.0,10.0,0.2,0.876045,0.732186
18,2.0,4.0,0.1,0.876045,0.732077
4,1.0,2.0,0.2,0.875241,0.732063
33,5.0,10.0,0.1,0.874598,0.731042
20,2.0,4.0,0.5,0.871543,0.730335
7,1.0,4.0,0.2,0.860932,0.720916
6,1.0,4.0,0.1,0.861093,0.720131
